In [1]:
print("Hello, World!")

Hello, World!


In [2]:
%pwd

'd:\\NEW Projects\\RAG-Based-Medical-ChatBot\\Experiment'

In [3]:
import os
os.chdir('D:/NEW Projects/RAG-Based-Medical-ChatBot')

In [4]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
# def load_files(data):
#     loader = DirectoryLoader(data, glob='*.pdf',loader_cls=PyPDFLoader)
#     documents = loader.load()
#     return documents

In [6]:
# extracted_data = load_files(data='Data/')


In [7]:
# def spilt_text(extracted_data):
#     text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
#     chunks = text_splitter.split_documents(extracted_data)
#     return chunks

In [8]:
# chunks = spilt_text(extracted_data)


In [9]:
# len(chunks)

In [10]:
# chunks[0].page_content

In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [12]:
def download_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [13]:
embeddings = download_embeddings()

C:\Users\trish\AppData\Local\Temp\ipykernel_15104\2372466182.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
d:\NEW Projects\RAG-Based-Medical-ChatBot\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


In [14]:
results = embeddings.embed_query("hii")
len(results)

384

In [15]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [16]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
NVIDIA_NIM = os.getenv("NVIDIA_NIM")
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["NVIDIA_NIM"] = NVIDIA_NIM

In [17]:
from pinecone import Pinecone, ServerlessSpec

# pc = Pinecone()
index_name = "medical-chatbot"

# pc.create_index(
#     name=index_name,
#     dimension=384,
#     metric="cosine",
#     spec=ServerlessSpec(
#         cloud="aws",
#         region="us-east-1"
#     )
# )

In [18]:
# from langchain_pinecone import PineconeVectorStore
# search=PineconeVectorStore.from_documents(chunks, embeddings, index_name=index_name)

In [19]:
from langchain_pinecone import PineconeVectorStore
search=PineconeVectorStore.from_existing_index(index_name=index_name, embedding=embeddings)

In [20]:
retriever = search.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [22]:
retrived_docs = retriever.invoke("What is diabetes?")
retrived_docs

[Document(metadata={'creationdate': '2006-10-16T20:19:33+02:00', 'creator': 'Adobe Acrobat 6.0', 'moddate': '2006-10-16T22:03:45+02:00', 'page': 1184.0, 'page_label': '1155', 'producer': 'PDFlib+PDI 6.0.3 (SunOS)', 'source': 'Data\\medicine-book.pdf', 'total_pages': 4505.0}, page_content='Description\nDiabetes mellitus is a chronic disease that causes\nserious health complications including renal (kidney)\nfailure, heart disease, stroke, and blindness.\nApproximately 17 million Americans have diabetes.\nUnfortunately, as many as one-half are unaware they\nhave it.\nBackground\nEvery cell in the human body needs energy in\norder to function. The body’s primary energy source\nis glucose, a simple sugar resulting from the digestion\nof foods containing carbohydrates (sugars and'),
 Document(metadata={'creationdate': '2006-10-16T20:19:33+02:00', 'creator': 'Adobe Acrobat 6.0', 'moddate': '2006-10-16T22:03:45+02:00', 'page': 1184.0, 'page_label': '1155', 'producer': 'PDFlib+PDI 6.0.3 (SunOS

In [23]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="meta/llama-3.1-70b-instruct",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_NIM"],
    temperature=0.2
)

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """You are a helpful RAG assistant for answering questions related to medical topics.
Use the following retrieved documents to provide accurate and concise answers to the user's questions. 
If you don't know the answer, say you don't know.

Retrieved context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [36]:
from langchain_core.prompts import ChatPromptTemplate

rewriteprompt = """
Conversation:
{chat_history}

Current question:
{input}

Rewrite the question so it is self-contained.
"""

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", rewriteprompt)
])

question_chain = rewrite_prompt | llm

In [31]:
chat_history = []
MAX_HISTORY = 5

In [38]:
result=question_chain.invoke({"input": "What is diabetes?", "chat_history": chat_history}).content
result

"What is the definition and explanation of diabetes, a common medical condition affecting the body's ability to regulate blood sugar levels?"

In [39]:
question_answering_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
qa_chain = create_retrieval_chain(retriever, question_answering_chain)

In [40]:
response = llm.invoke("Hello")
response.content

'Hello. How can I assist you today?'

In [41]:
question = "What is diabetes?"
history_text = "\n".join([
    f"User: {chat['user']}\nAssistant: {chat['assistant']}"
    for chat in chat_history
])

response = qa_chain.invoke({
    "input": question_chain.invoke({"input": question, "chat_history": chat_history}).content,
    "chat_history": chat_history
})

print(response["answer"])

Diabetes mellitus is a chronic disease that affects the body's ability to regulate blood sugar levels. It is a condition in which the body cannot effectively use sugar (glucose) in the blood. This occurs when the body either does not produce enough insulin, a hormone that helps to convert glucose into energy, or cannot use the insulin it produces.

As a result, blood sugar levels can become very high, leading to a range of complications if left untreated. These complications can include damage to the eyes, kidneys, heart, nerves, blood vessels, and other organs.

In a normal body, glucose is the primary energy source for cells. When we eat foods containing carbohydrates, they are broken down into glucose, which is then absorbed into the bloodstream. Insulin, produced by the pancreas, helps to facilitate the entry of glucose into cells, where it can be used for energy.

In people with diabetes, this process is disrupted, leading to high blood sugar levels. There are different types of d

In [39]:
chat_history.append({
    "user": question,
    "assistant": response["answer"]
})
chat_history = chat_history[-MAX_HISTORY:]

In [40]:
question = "What are its symptoms?"

response = qa_chain.invoke({
    "input": question,
    "chat_history": chat_history
})

print(response["answer"])

The symptoms you're referring to seem to be related to a few different conditions. Based on the provided context, I'll break down the symptoms for each condition:

1. **Systemic lupus erythematosus (SLE):** 
   - Fever
   - Chills
   - Fatigue
   - Weight loss
   - Skin rashes (particularly the classic 'butterfly' rash on the face)
   - Vasculitis
   - Polyarthralgia
   - Patchy hair loss
   - Sores in the mouth or nose
   - Lymph-node enlargement
   - Gastric problems
   - Irregular periods in women
   - Cardiopulmonary problems
   - Urinary problems

2. **Another condition ( possibly related to liver or gastrointestinal issues):**
   - Blood in stools
   - Bloody vomit
   - Vomiting material that looks like coffee grounds
   - Excessive tiredness
   - Unusual bleeding or bruising
   - Itching
   - Lack of energy
   - Loss of appetite
   - Pain in the upper right part of the stomach
   - Yellowing of the skin or eyes
   - Flu-like symptoms
   - Rash
   - Pale skin
   - Unexplained wei

In [43]:
chat_history

[{'user': 'What is diabetes?',
  'assistant': 'Diabetes mellitus is a condition in which the pancreas no longer produces enough insulin or cells stop responding to the insulin that is produced, so that glucose in the blood cannot be absorbed into the cells of the body.'}]